In [2]:
# Cell 1: Import Library & Setup Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("Library berhasil diimpor!")

Library berhasil diimpor!


In [3]:
# Cell 2: Data Gathering - Memuat Semua Dataset

customers_df = pd.read_csv('customers_dataset.csv')
geolocation_df = pd.read_csv('geolocation_dataset.csv')
orders_df = pd.read_csv('orders_dataset.csv')
order_items_df = pd.read_csv('order_items_dataset.csv')
order_payments_df = pd.read_csv('order_payments_dataset.csv')
order_reviews_df = pd.read_csv('order_reviews_dataset.csv')
products_df = pd.read_csv('products_dataset.csv')
sellers_df = pd.read_csv('sellers_dataset.csv')
category_translation_df = pd.read_csv('product_category_name_translation.csv')

print("Semua dataset berhasil dimuat!")

Semua dataset berhasil dimuat!


In [4]:
# Cell 3: Data Assessing - Memeriksa Tipe Data, Missing Values, dan Duplikasi

dfs = {
    'customers': customers_df,
    'orders': orders_df,
    'order_items': order_items_df,
    'order_payments': order_payments_df,
    'order_reviews': order_reviews_df,
    'products': products_df,
    'sellers': sellers_df,
    'geolocation': geolocation_df,
    'category_translation': category_translation_df
}

print("=== RINGKASAN DATA ASSESSING ===\n")
for name, df in dfs.items():
    print(f"--- DataFrame: {name} ---")
    print(f"Jumlah Baris & Kolom: {df.shape}")
    print(f"Jumlah Duplikat: {df.duplicated().sum()}")
    nulls = df.isnull().sum()
    print("Missing Values:")
    print(nulls[nulls > 0] if nulls.sum() > 0 else "Tidak ada missing value")
    print("\n" + "="*40 + "\n")

=== RINGKASAN DATA ASSESSING ===

--- DataFrame: customers ---
Jumlah Baris & Kolom: (99441, 5)
Jumlah Duplikat: 0
Missing Values:
Tidak ada missing value


--- DataFrame: orders ---
Jumlah Baris & Kolom: (99441, 8)
Jumlah Duplikat: 0
Missing Values:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64


--- DataFrame: order_items ---
Jumlah Baris & Kolom: (112650, 7)
Jumlah Duplikat: 0
Missing Values:
Tidak ada missing value


--- DataFrame: order_payments ---
Jumlah Baris & Kolom: (103886, 5)
Jumlah Duplikat: 0
Missing Values:
Tidak ada missing value


--- DataFrame: order_reviews ---
Jumlah Baris & Kolom: (99224, 7)
Jumlah Duplikat: 0
Missing Values:
review_comment_title      87656
review_comment_message    58247
dtype: int64


--- DataFrame: products ---
Jumlah Baris & Kolom: (32951, 9)
Jumlah Duplikat: 0
Missing Values:
product_category_name         610
product_name_lenght           610
product_description_l

In [5]:
# Cell 4: Data Cleaning - Konversi Tipe Data & Handling Missing Value

# 1. Konversi kolom tanggal pada orders_df ke datetime
datetime_cols_orders = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in datetime_cols_orders:
    orders_df[col] = pd.to_datetime(orders_df[col])

# 2. Konversi kolom tanggal pada order_items_df & order_reviews_df
order_items_df['shipping_limit_date'] = pd.to_datetime(order_items_df['shipping_limit_date'])
order_reviews_df['review_creation_date'] = pd.to_datetime(order_reviews_df['review_creation_date'])
order_reviews_df['review_answer_timestamp'] = pd.to_datetime(order_reviews_df['review_answer_timestamp'])

# 3. Penerjemahan kategori produk ke bahasa Inggris & penanganan missing values
products_df['product_category_name'] = products_df['product_category_name'].fillna('unknown')
products_df = products_df.merge(category_translation_df, on='product_category_name', how='left')
products_df['product_category_name_english'] = products_df['product_category_name_english'].fillna('unknown')

# 4. Pengisian missing values pada teks ulasan
order_reviews_df['review_comment_title'] = order_reviews_df['review_comment_title'].fillna('No Title')
order_reviews_df['review_comment_message'] = order_reviews_df['review_comment_message'].fillna('No Message')

print("Data Cleaning berhasil!")

Data Cleaning berhasil!


In [6]:
# Cell 5 (Revisi): Data Cleaning & Agregasi Berstruktur

# 1. Filter order delivered
main_df = orders_df[orders_df['order_status'] == 'delivered'].copy()

# 2. Agregasi Pembayaran per order_id (mencegah duplikasi pembayaran)
payments_agg = order_payments_df.groupby('order_id').agg(
    total_payment_value=('payment_value', 'sum'),
    payment_type_main=('payment_type', lambda x: x.iloc[0]) # Metode bayar utama
).reset_index()

# 3. Agregasi Item & Produk per order_id
items_with_prod = order_items_df.merge(products_df[['product_id', 'product_category_name_english']], on='product_id', how='left')

items_agg = items_with_prod.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    total_items=('order_item_id', 'count'),
    product_category_main=('product_category_name_english', lambda x: x.iloc[0] if len(x) > 0 else 'unknown')
).reset_index()

# 4. Agregasi Review Score per order_id
reviews_agg = order_reviews_df.groupby('order_id')['review_score'].mean().reset_index()

# 5. Merge seluruh data ke level order_id (1 baris = 1 transaksi unik)
main_df = main_df.merge(customers_df[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']], on='customer_id', how='left')
main_df = main_df.merge(payments_agg, on='order_id', how='left')
main_df = main_df.merge(items_agg, on='order_id', how='left')
main_df = main_df.merge(reviews_agg, on='order_id', how='left')

# 6. Clean Missing Values & Duplikat
main_df['review_score'] = main_df['review_score'].fillna(main_df['review_score'].mean())
main_df['product_category_main'] = main_df['product_category_main'].fillna('unknown')
main_df.drop_duplicates(inplace=True)

# Bulatkan kolom berformat desimal keuangan ke 2 digit belakang koma
main_df['total_payment_value'] = main_df['total_payment_value'].round(2)
main_df['total_price'] = main_df['total_price'].round(2)
main_df['total_freight'] = main_df['total_freight'].round(2)
main_df['review_score'] = main_df['review_score'].round(1)

# Simpan kembali ke main_data.csv
main_df.to_csv('main_data.csv', index=False)
print("File main_data.csv berhasil diperbarui dengan desimal rapi!")

print(f"Data Cleaning Selesai! Total baris bersih: {len(main_df):,}")
print(f"Total Revenue Akurat: R$ {main_df['total_payment_value'].sum():,.2f}")

File main_data.csv berhasil diperbarui dengan desimal rapi!
Data Cleaning Selesai! Total baris bersih: 96,478
Total Revenue Akurat: R$ 15,422,461.77


In [15]:
%%writefile dashboard.py
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st

# Setup Konfigurasi Halaman Streamlit
st.set_page_config(
    page_title="E-Commerce Data Dashboard",
    page_icon="🛒",
    layout="wide"
)

# Load Data dengan Caching & Handling Nama Kolom
@st.cache_data
def load_data():
    df = pd.read_csv("main_data.csv")
    df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

    # Standarisasi kolom jika terdapat perbedaan penamaan
    if 'payment_value' in df.columns and 'total_payment_value' not in df.columns:
        df.rename(columns={'payment_value': 'total_payment_value'}, inplace=True)
    if 'product_category_name_english' in df.columns and 'product_category_main' not in df.columns:
        df.rename(columns={'product_category_name_english': 'product_category_main'}, inplace=True)

    return df

df = load_data()

# Header Dashboard
st.title("🛒 E-Commerce Public Data Dashboard")
st.markdown("Dashboard interaktif untuk menganalisis kinerja penjualan, kategori produk, dan lokasi pelanggan.")

# Sidebar Filter Tanggal
min_date = df['order_purchase_timestamp'].min().date()
max_date = df['order_purchase_timestamp'].max().date()

with st.sidebar:
    st.header("Filter Data")
    date_range = st.date_input(
        label="Rentang Waktu",
        min_value=min_date,
        max_value=max_date,
        value=[min_date, max_date]
    )

# Validasi Pilihan Tanggal
if isinstance(date_range, (list, tuple)) and len(date_range) == 2:
    start_date, end_date = date_range
elif isinstance(date_range, (list, tuple)) and len(date_range) == 1:
    start_date = end_date = date_range[0]
else:
    start_date = end_date = date_range

# Filter Data
main_filtered_df = df[
    (df['order_purchase_timestamp'].dt.date >= start_date) &
    (df['order_purchase_timestamp'].dt.date <= end_date)
]

# Matriks Ringkasan (KPI)
col1, col2, col3 = st.columns(3)
with col1:
    st.metric("Total Orders", value=f"{len(main_filtered_df):,}")
with col2:
    st.metric("Total Revenue", value=f"R$ {main_filtered_df['total_payment_value'].sum():,.2f}")
with col3:
    st.metric("Average Review Score", value=f"{main_filtered_df['review_score'].mean():.2f}")

st.divider()

# Grafik 1: Tren Penjualan Bulanan
st.subheader("📈 Tren Pendapatan Penjualan")
monthly_revenue = main_filtered_df.resample(rule='M', on='order_purchase_timestamp')['total_payment_value'].sum().reset_index()
monthly_revenue['order_purchase_timestamp'] = monthly_revenue['order_purchase_timestamp'].dt.strftime('%Y-%m')

fig_trend, ax_trend = plt.subplots(figsize=(12, 4))
sns.lineplot(data=monthly_revenue, x='order_purchase_timestamp', y='total_payment_value', marker='o', color='#1f77b4', ax=ax_trend)
ax_trend.set_xlabel("Bulan")
ax_trend.set_ylabel("Total Pendapatan (R$)")
plt.xticks(rotation=45)
st.pyplot(fig_trend)

st.divider()

# Grafik 2 & 3: Top Kategori & Wilayah Pelanggan
col_left, col_right = st.columns(2)

with col_left:
    st.subheader("📦 Top 5 Kategori Produk Terlaris")
    top_cat = main_filtered_df.groupby("product_category_main")["total_payment_value"].sum().sort_values(ascending=False).head(5).reset_index()
    fig_cat, ax_cat = plt.subplots(figsize=(6, 4))
    sns.barplot(data=top_cat, y="product_category_main", x="total_payment_value", hue="product_category_main", palette="Blues_r", legend=False, ax=ax_cat)
    ax_cat.set_xlabel("Total Sales (R$)")
    ax_cat.set_ylabel("Kategori Produk")
    st.pyplot(fig_cat)

with col_right:
    st.subheader("📍 Top 5 Wilayah Pelanggan (State)")
    top_state = main_filtered_df["customer_state"].value_counts().head(5).reset_index()
    top_state.columns = ["customer_state", "count"]
    fig_state, ax_state = plt.subplots(figsize=(6, 4))
    sns.barplot(data=top_state, x="customer_state", y="count", hue="customer_state", palette="Greens_r", legend=False, ax=ax_state)
    ax_state.set_xlabel("Negara Bagian (State)")
    ax_state.set_ylabel("Jumlah Transaksi")
    st.pyplot(fig_state)

st.divider()
st.subheader("📄 Pratinjau Data Transaksi")
st.dataframe(main_filtered_df.head(10))

Overwriting dashboard.py


In [16]:
# Cell: Membuat file requirements.txt
%%writefile requirements.txt
pandas
matplotlib
seaborn
streamlit

Writing requirements.txt


In [27]:
# Cell: Membuat file README.md
%%writefile README.md
# 🛒 E-Commerce Data Analysis & Dashboard

## Deskripsi Proyek
Proyek ini merupakan analisis data eksploratif (EDA) dan pembuatan dashboard interaktif berbasis **Streamlit** untuk menganalisis kinerja transaksi e-commerce, tren penjualan bulanan, kategori produk terlaris, serta distribusi lokasi pelanggan.

## Struktur Direktori Proyek
submission/
├── dashboard/
│   ├── dashboard.py
│   └── main_data.csv
├── notebook.ipynb
├── README.md
└── requirements.txt
git clone https://github.com/rasyidd18/Asah.git
cd Asah
pip install -r requirements.txt
streamlit run dashboard/dashboard.py

Overwriting README.md


In [18]:
# Cell: Menata struktur folder submission
import os
import shutil

# Buat folder dashboard
os.makedirs('dashboard', exist_ok=True)

# Salin file ke dalam folder dashboard
shutil.copy('dashboard.py', 'dashboard/dashboard.py')
shutil.copy('main_data.csv', 'dashboard/main_data.csv')

print("Folder 'dashboard' berhasil dibuat dan diisi!")

Folder 'dashboard' berhasil dibuat dan diisi!
